In [2]:
from datasets import load_dataset

ds = load_dataset("rpmon/fma-genre-classification")

c:\Users\tim7m\OneDrive\Desktop\gp5\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\tim7m\OneDrive\Desktop\gp5\venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\tim7m\.cache\huggingface\hub\datasets--rpmon--fma-genre-classification. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In o

In [10]:
df_train = ds['train'].to_pandas()
df_val = ds['validation'].to_pandas()

df_train.head()

,audio,genre,track_id,title,artist
0,{'bytes': b'ID3\x04\x00\x00\x00\x00\x041TIT2\x...,0,55183,Intro To Horror,Dylan Palme
1,{'bytes': b'ID3\x04\x00\x00\x00\x00\x05\x1eTIT...,6,120184,Lavender Hip Mob,Lee Rosevere
2,{'bytes': b'ID3\x04\x00\x00\x00\x00\x04aTIT2\x...,7,85816,Break It Now,Radio 421
3,{'bytes': b'ID3\x04\x00\x00\x00\x00\x06\x03TIT...,5,36277,"Restaurant Concert, Song 2",Demiran Ćerimović and His Orkestar
4,{'bytes': b'ID3\x04\x00\x00\x00\x00\x04VTIT2\x...,7,56552,Celebration,Krebs


In [ ]:
import io
import os
import numpy as np
from pydub import AudioSegment
import soundfile as sf


for df, out_dir in zip(
    [df_train, df_val],
    ["data/audio_train", "data/audio_val"]
):

    os.makedirs(out_dir, exist_ok=True)

    for _, row in df.iterrows():

        track_id = row["track_id"]
        out_path = f"{out_dir}/{track_id}.wav"

        if os.path.exists(out_path):
            continue

        try:
            audio_bytes = row["audio"]["bytes"]

            audio = AudioSegment.from_file(io.BytesIO(audio_bytes), format="mp3")

            samples = np.array(audio.get_array_of_samples())

            if audio.channels > 1:
                samples = samples.reshape((-1, audio.channels)).mean(axis=1) # усреднение по каналу

            samples = samples.astype(np.float32)
            samples /= (np.max(np.abs(samples)) + 1e-9)

            sf.write(out_path, samples, audio.frame_rate)

        except Exception as e:
            print(track_id, e)

In [ ]:

path = "data_csv/audio_val"

num_files = sum(
    len(files)
    for _, _, files in os.walk(path)
)

print(f"Количество файлов: {num_files}")

Количество файлов: 1600


In [ ]:
path = "data_csv/audio_train"

num_files = sum(
    len(files)
    for _, _, files in os.walk(path)
)

print(f"Количество файлов: {num_files}")